# Fig: mnist

project = ```iP-VAE```, host = ```any```, device = ```any```

**Motivation**: <br>


In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_IterativeVAE'))
from figures.convergence import plot_convergence
from figures.imgs import plot_weights
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

In [2]:
device_idx = 1
device = f'cuda:{device_idx}'

print(f"device: {device}  ———  host: {os.uname().nodename}")

device: cuda:1  ———  host: mach

## Results & Figs dir

In [3]:
pal_models = get_palette_models()

results_dir = pjoin(tmp_dir, 'results_2025-03')
figs_dir = pjoin(fig_base_dir, 'results_2025-03')

os.makedirs(results_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)

len(os.listdir(results_dir)), len(os.listdir(figs_dir))

(369, 0)

## MNIST / FashionMNIST

In [4]:
files = [f for f in os.listdir(results_dir) if 'MNIST' in f]
files = sorted(files, key=alphanum_sort_key)

len(files)

12

In [5]:
info_keys = [
    'dataset', 'type', 'latent_act', 'model_str',
    'seq_len', 'kl_beta', 'n_latents',
]

df = collections.defaultdict(list)
for f in tqdm(files):
    is_amortized = 'amort' in f
    try:
        load = np.load(
            pjoin(results_dir, f),
            allow_pickle=True,
        ).item()
    except EOFError:
        print(f"EOFError:\n{f}")
        continue

    info, results = load['info'], load['results']
    info['n_latents'] = info['n_latents'][0]

    # info
    for k in info_keys:
        df[k].append(info.get(k, None))
    df['amort'].append(is_amortized)

    # results
    vals = {
        'r2': results['results_xtract']['vld']['r2'][-1],
        'mse': results['results_xtract']['vld']['mse'][-1],
        '%-zeros': np.mean(results['results_xtract']['vld']['samples'][:, -1, :] == 0.0),
        'accu': results['clf_accuracy']['vld'][1000],
    }
    for k, v in vals.items():
        df[k].append(v)
df = pd.DataFrame(df)
df.shape

100%|███████████████████████████████████████████| 12/12 [00:05<00:00,  2.26it/s]


(12, 12)

In [6]:
df

,dataset,type,latent_act,model_str,seq_len,kl_beta,n_latents,amort,r2,mse,%-zeros,accu
0,FashionMNIST,poisson,None,poisson,16,8.0,512,False,0.820732,12.844995,0.860020,0.8654
1,FashionMNIST,poisson,None,poisson,16,16.0,512,False,0.809397,13.560640,0.878674,0.8624
2,FashionMNIST,poisson,None,poisson,16,32.0,512,False,0.806954,13.539106,0.912639,0.8599
3,FashionMNIST,poisson,None,poisson,32,32.0,512,False,0.851756,10.169729,0.812223,0.8705
4,FashionMNIST,poisson,None,poisson,64,64.0,512,False,0.888382,7.481263,0.783917,0.8733
5,FashionMNIST,poisson,None,poisson,64,128.0,512,False,0.879650,8.085887,0.801067,0.8712
6,FashionMNIST,poisson,None,poisson,128,64.0,512,False,0.951932,3.088070,0.675432,0.8647
7,FashionMNIST,poisson,None,poisson,128,96.0,512,False,0.947821,3.354098,0.705460,0.8670
8,FashionMNIST,poisson,None,poisson,128,128.0,512,False,0.944319,3.587626,0.719014,0.8687
9,FashionMNIST,poisson,None,poisson,128,256.0,512,False,0.933389,4.299773,0.776341,0.8727
